In [ ]:
# Purpose - plots fields and cuts for a single file

findex = 9 #63 #113 #121 #141 #130 #121

step_choice = "except_last"

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps, "penultimate": [run.steps[-2]], "except_last": run.steps[:-1]}

my_steps = tstep_choices[step_choice]
boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = False)

In [ ]:
rec_rate_values = []

for t in range(len(my_steps)):
    roi1 = fq_cut(d.fields, t, [0, run.xlim / 4, 0, run.ylim])
    roi2 = fq_cut(d.fields, t, [3 * run.xlim / 4, run.xlim, 0, run.ylim])
    rec_rate = (roi1.ez.mean() + roi2.ez.mean()) / 2 / (run.beta_A * run.B0)
    rec_rate_values.append(rec_rate.values)

# Convert to numpy array for easier plotting
rec_rate_values = np.array(rec_rate_values)

# Plot the reconnection rate over time
plt.figure()
plt.plot(my_steps, rec_rate_values, label='Reconnection Rate')
plt.xlabel('Time Step')
plt.ylabel('Reconnection Rate')
plt.title(boilerplate)
plt.legend()
plt.show()

print(rec_rate.values)

In [ ]:
# Choose Regions of Interest

xlim = run.xlim
ylim = run.ylim

#rect = [xlim / 2 - 4, xlim / 2 + 4, ylim / 2 - 10, ylim / 2 + 10]
#rect = [0, xlim, 0, ylim]
#rect = [50, 70, 130, 190]
rect = [0, xlim, 0, ylim]

tchosen = 0
roi = fq_cut(d.fields, tchosen, rect)

#xcut_y = 164.4
xcut_y = ylim / 10 #5950
xcut_y = 166
xcut_y = 120
ycut_x = xlim / 2

xstrip = fq_xcut(roi, xcut_y)
ystrip = fq_ycut(roi, ycut_x)

In [ ]:
if True:
    wholebox = fq_cut(d.fields, 39, [0, run.xlim, 0, run.ylim])

    rcut = fq_ycut(wholebox, run.xlim / 4)

    avg_velx_rcut = np.mean(rcut.velx.values)
    print(f"Avg rec rate: {avg_velx_rcut / run.beta_A:.2f}")

    plt.figure()
    plt.plot(rcut.y, rcut.velx / run.beta_A)
    plt.axhline(y=avg_velx_rcut / run.beta_A, color='r', linestyle='--', label=f"Mean: {avg_velx_rcut / run.beta_A:.2f}")
    plt.xlabel('$y$ ($c/\\omega_{p}$)')
    plt.xlim(0, run.ylim)
    plt.ylabel('$v_x / v_A$')
    plt.title(boilerplate + '\n$R$ evaluated at $x = L_x / 4$')
    plt.legend()
    plt.show()

In [ ]:
#roi = fq_cut(d.fields, 0, [80, 160, 0, 200])
roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim])

#roi = fq_cut(d.fields, 0, [55, 65, 110, 130])

# Create a subplot with one plot
fig, ax = plt.subplots()

plt.title(f"{run.boilerplate}")
plot_basic(ax, run, roi, sdens = 0.5, cutoff = 5)

# Show the plot
plt.show()

In [ ]:
print(run.output_interval)
print(run.istep)
print(run.B0)
print(run.c_omp)

In [ ]:
def compute_Az(roi):    
    return (np.cumsum(roi.bx, axis=0) - np.cumsum(roi.by, axis=1)) * run.istep #properly should multiply this by the spacing

In [ ]:
# Plot Regions

delta = 80
rect = [run.xlim / 2 - delta, run.xlim / 2 + delta, 0, 500]
wholebox = [0, run.xlim, 0, run.ylim]

rect = wholebox

cs = fq_cut(d.fields, 0, wholebox)

ez_treal = cs.bx * cs.vely - cs.by * cs.velx 
ez_nontreal = cs.ez - ez_treal 
ez_tot = cs.ez
JdotE = cs.ex * cs.jx + cs.ey * cs.jy + cs.ez * cs.jz

Az = compute_Az(cs)

cs['ez_nonideal'] = cs.ez + cs.velx * cs.by - cs.vely * cs.bx
cs['ez_ideal'] = cs.ez - cs.ez_nonideal

dens = cs.dens1 + cs.dens2

esq, bsq, chi = sqrd(cs) 

adj_factor = run.beta_A

bsq_adj =  adj_factor ** 2 * bsq
chi_adj = (bsq_adj - esq) / (bsq_adj + esq)

v_A = run.beta_A

plt.figure()

# --------------STREAMLINES--------------

#plot_streamline(plt, cs, cs.bx, cs.by, 2, 'white') 
#plot_streamline(plt, cs, cs.velx, cs.vely, 2, 'black') 

# --------------CONTOURS--------------

plt.contour(cs.x, cs.y, chi_adj, [0], colors = 'green')

Ainc = (0.16 * run.beta_A * run.cc) * run.B0 * run.output_interval

#plt.contour(cs.x, cs.y, Az, [1, 1 + Ainc], colors = 'green')

# --------------AREA PLOTS--------------
# SCALARS

#plot_fq3(cs, cs.dens, vabsmax = 3)
#plot_fq3(cs, cs.epsilon - 1, vabsmax = 2 * run.sigma, cmap = 'seismic')

# FIELDS

#plot_fq3(cs, cs.ex / B0, "$E_x / B_0$", vabsmax = 0.15, cmap = 'seismic')
#plot_fq3(cs, cs.ey / B0, "$E_y / B_0$", vabsmax = 0.3)
#plot_fq3(cs, cs.ez / B0, "$E_z / B_0$", vabsmax = 0.15)

#plot_fq3(cs, cs.by, vabsmax = 0.15)
#plot_fq3(cs, Az)

# NONIDEAL EFFECTS

#plot_fq3(cs, cs.ez / run.B0, vabsmax = 0.15)
#plot_fq3(cs, cs.ez_ideal / run.B0, cmap = "seismic", vabsmax = 0.1)
#plot_fq3(cs, cs.ez_nonideal / run.B0, cmap = "seismic", vabsmax = 0.1)
#plot_fq3(cs, cs.jz, f"$J_z$", vabsmax = 1e-4, cmap = 'seismic')
#plot_fq3(cs, cs.jz * cs.ez, "JzEz", vabsmax = 1e-7)
#plot_fq3(cs, cs.ez_nonideal / cs.jz, vabsmax = 20)

# VELOCITY

#plot_fq3(cs, cs.velx / v_A, boilerplate + "$v_x / v_A$", vabsmax = 0.6)
#plot_fq3(cs, cs.velx / v_A, vabsmax = 1.5, cmap = 'seismic')
plot_fq3(cs, cs.dens * cs.velx / v_A, vabsmax = 1.5, cmap = 'seismic')

# --------------PLASMOID IDENTIFIERS--------------

# Find points where ystrip.bx switches sign
sign_change_indices = np.where(np.diff(np.sign(ystrip.bx)) > 0)[0]

plt.title(run.boilerplate + f"\n t = {run.steps[0]} $\sigma =$ {run.sigma}")
plt.vlines(ycut_x, rect[2], rect[3], colors = 'yellow', linestyle = '--')
plt.hlines(xcut_y, rect[0], rect[1], colors = 'yellow', linestyle = '--')
plt.xlim(rect[:2])
plt.ylim(rect[2:])
plt.show()

In [ ]:
print(sign_change_indices)

In [ ]:
# X-strip 

strip = xstrip

ez_ideal = strip.bx * strip.vely - strip.by * strip.velx 
ez_nonideal = strip.ez - ez_treal 
ez_tot = strip.ez
dens = strip.dens1 + strip.dens2

JdotE = strip.jx * strip.ex + strip.jy * strip.ey + strip.jz * strip.ez

esq, bsq, chi = sqrd(strip)


jadj = 200

plt.figure()
#plt.title(boilerplate + f"t = {tchosen}, y = {xcut_y}")

#plt.plot(strip.x, ez_nontreal / (cc * B0), label = "$E_{z, ni} / c B_0$", color = '#d62728')
#plt.plot(strip.x, ez_treal / (cc * B0), label = "ez_treal / c B_0")
#plt.plot(strip.x, strip.ez / (cc * B0), label = "$E_z / c B_0$", color = '#1f77b4') #ez
#plt.plot(strip.x, bsq / B0 ** 2, label = "bsq / B0^2") #bsq
#plt.plot(strip.x, strip.jz / B0 * run.c_omp, label = '$J_z (c / \omega_p) / B_0$', color = '#2ca02c') #jz

plt.plot(strip.x, strip.by / B0, label = "$B_y / B_0$", color = '#ff7f0e')
plt.plot(strip.x, strip.epsilon, label = "$\epsilon$")
#plt.plot(strip.x, np.sqrt(bsq) / B0, label = "$B / B_0$", color = '#ff7f0e')
#plt.plot(strip.x, bsq / B0 ** 2, label = "$B^2 / B_0^2$", color = '#ff7f0e')
#plt.plot(strip.x, strip.bz / B0, label = "Bz / B0")
#plt.plot(strip.x, chi, label = '$\chi$')
#plt.plot(strip.x, chi_adj, label = '$\chi_{adj}$')
#plt.plot(strip.x, strip.velx / v_A, label = "$v_x / v_A$", color = '#9467bd')
#plt.plot(strip.x, strip.vely / v_A, label = "$v_y / v_A$", color = 'magenta')
#plt.plot(strip.x, np.sqrt(strip.velx ** 2 + strip.vely ** 2) / v_A, label = "$v / v_A$", color = '#9467bd')
plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.axvline(x = ycut_x, color = 'black', linestyle = '--')

#plt.plot(strip.x, strip.pb, label = "$p_B$")
#plt.plot(strip.x, strip.px, label = "$p_x$")

#plt.plot(strip.x, strip.pb + strip.px, label = "total pressure in x dir")

nbx = strip.bx / np.sqrt(bsq)
nby = strip.by / np.sqrt(bsq)
nbz = strip.bz / np.sqrt(bsq)
#ppara = strip.px * nbx + strip.py * nbx + strip.pz * nbz

#e_cr = (5 * strip.p + 2) / (5 * strip.p + 5)
#epsilon = 1 - 0.5 * (2 * ppara - strip.p)

#plt.plot(strip.x, e_cr, label = "$e_{cr}$")
#plt.plot(strip.x, epsilon, label = "$\epsilon$")

#plt.plot(strip.x, strip.px / strip.pb, label = "$beta$")

# New
#plt.plot(strip.x, strip.ex / (cc * B0), label = "$E_x / c B_0$")
#plt.plot(strip.x, strip.velz / v_A, label = "velz / v_A")
#plt.plot(strip.x, -strip.by * strip.velz / (cc * B0), label = "E_xni")
#plt.plot(strip.x, strip.dens1 / 16, label = 'n(e-)')
#plt.plot(strip.x, strip.dens2 / 16, label = 'n(e+)')
#plt.plot(strip.x, (strip.dens1 + strip.dens2)/ 16, label = '$n$')
#plt.plot(strip.x, (strip.dens1 + strip.dens2)/ 16 * (np.sqrt(bsq) / B0), label = '$(n / n_0) * (B / B_0)$', color = 'grey')
#plt.plot(strip.x, jadj * strip.jz / (dens * B0) * 16, label = f'{jadj} * j / (n / n0)')

#plt.plot(strip.x, (strip.dens2 - strip.dens1) / 16, label = 'rho')
#plt.plot(strip.x, jadj * strip.jx / B0, label = f'{jadj} * jx / B0')

#plt.plot(strip.x, ez_nonideal / strip.jz * 1/10, label = "eta_eff")
#plt.title(f"$\sigma =$ {run.sigma}")
plt.title(boilerplate + f"x strip y = {xcut_y}")
plt.xlabel('$x$ [$c/\omega_{p}$]')
plt.ylabel('Quantity')
plt.legend(loc='upper left')

plt.show()

In [ ]:
que = M_SM.values

In [ ]:
plt.figure()
plt.plot(strip.x, M_SM, label="M_SM")
plt.axhline(y=0, color='black', linestyle='--')
plt.axvline(x=ycut_x, color='black', linestyle='--')
plt.title(boilerplate + f" x strip y = {xcut_y}")
plt.xlabel('$x$ [$c/\\omega_{p}$]')
plt.ylabel('M_SM')
plt.legend(loc='center left')
plt.show()

In [ ]:
# Check if any element in in_sqrt is negative
negative_indexer = (in_sqrt < 0).compute()
negative_elements = in_sqrt[negative_indexer]

if negative_elements.size > 0:
    print("There are negative elements in in_sqrt.")
else:
    print("There are no negative elements in in_sqrt.")

In [ ]:
from scipy.integrate import simps

# Define the interval for integration
x_start, x_end = 332, 418
x_start, x_end = 340, 410


# Extract the relevant portion of the data
interval = (strip.x >= x_start) & (strip.x <= x_end)
x_values = strip.x[interval]
bsq_values = bsq[interval] / B0 ** 2

# Perform the integration
average_result = simps(bsq_values, x_values) / (x_end - x_start)

print(f"Integral of B^2 / B_0^2 over the interval [{x_start}, {x_end}] is {average_result}")

In [ ]:
print(list(strip.data_vars))

In [ ]:
# Y-strip 

strip = ystrip

strip['ez_ideal'] = strip.bx * strip.vely - strip.by * strip.velx 
strip['ez_nonideal'] = strip.ez - strip.ez_ideal 
ez_tot = strip.ez

JdotE = strip.jx * strip.ex + strip.jy * strip.ey + strip.jz * strip.ez
dens = strip.dens1 + strip.dens2

esq, bsq, chi = sqrd(strip)

adj_factor = run.beta_A ** 2


bsq_adj = adj_factor ** 2 * bsq
chi_adj = (bsq_adj - esq) / (bsq_adj + esq)

sign_change = np.diff(np.sign(chi_adj))
#y_gate_index = np.where(sign_change > 0)[-1][-1]

jadj = 200

plt.figure()
plt.title(boilerplate + f"\n t = {tchosen}, x = {ycut_x}")
#plt.plot(strip.y, ez_nontreal / (cc * B0), label = "ez_nontreal / c B_0")
#plt.plot(strip.y, ez_treal / (cc * B0), label = "ez_treal / c B_0")
#plt.plot(strip.y, strip.ez / (cc * B0), label = "$E_z / c B_0$", color = '#1f77b4') #ez
#plt.plot(strip.y, bsq / B0 ** 2, label = "bsq / B0^2") #bsq
#plt.plot(strip.y, strip.jz / B0 * run.c_omp, label = '$J_z (c / \omega_p) / B_0$', color = '#2ca02c') #jz
#plt.plot(strip.y, strip.bx / B0, label = "$B_x / B_0$", color = '#ff7f0e')
#plt.plot(strip.y, strip.bz / B0, label = "Bz / B0")
#plt.plot(strip.y, chi, label = '$\chi$')
#plt.plot(strip.y, chi_adj, label = '$\chi_{adj}$', color = 'orange')
plt.plot(strip.y, strip.vely / run.beta_A, label = "$v_y / v_A$", color = '#9467bd')
#plt.plot(strip.y, strip.velx / run.beta_A, label = "$v_y / v_A$", color = 'red')
#plt.plot(strip.y, strip.dens / 16, label = '$n$')

#plt.plot(strip.y, strip.ez_nonideal / strip.jz, label = r"$\eta$", color = 'orange')


avg_velx_v_A = np.mean(strip.velx.values / run.beta_A)
print(avg_velx_v_A)

avg_velx_v_A = np.mean(dens.values / 16)
print(avg_velx_v_A)

plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.axvline(x = xcut_y, color = 'black', linestyle = '--')

#plt.plot(strip.y, jadj * ez_nontreal * strip.jz / (cc * B0 ** 2), label = f"{jadj} * ezni jz")
#plt.plot(strip.y, ez_nontreal / (cc * B0), label = "$E_{zni}$ / $c B_0$", color = '#d62728')
#plt.axvline(x = strip.y[y_gate_index + 1], color = 'orange', linestyle = '--')

#plt.plot(strip.x, ez_nontreal / strip.jz * 1/10, label = "eta_eff")
#plt.title(f"$\sigma =$ {run.sigma}")
plt.title(boilerplate + f"\n y strip x = {ycut_x}")
plt.legend(loc = 'lower right')
plt.xlabel('$y$ [$c/\omega_{p}$]')
plt.ylabel('Quantity')
#plt.ylim([-1.5, 5])
plt.show()

In [ ]:
print(y_gate[-1][-1])

In [ ]:
p